In [ ]:
print("Ok, let's start with RAG demo")

In [ ]:
# Get file
from pathlib import Path
file_path = Path("../data/information.txt").resolve()
print(f"File path: {file_path}")

In [ ]:
# Load document
from langchain_community.document_loaders import TextLoader
loader = TextLoader(file_path, encoding="utf-8")
documents = loader.load()
print(f"Loaded {len(documents)} document(s)")

In [ ]:
documents[0]

In [ ]:
# Create chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)
chunks = text_splitter.split_documents(documents=documents)
print(f"Created {len(chunks)} chunk(s)")

In [ ]:
# create embeddings
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv
load_dotenv() # load environment variables from .env file
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=1024)


In [ ]:
embeddings = embedding_model.embed_documents([chunk.page_content for chunk in chunks])
embeddings

In [ ]:
# Create vectorstore 
from langchain_community.vectorstores import FAISS
vector_store = FAISS.from_documents(documents=chunks, embedding=embedding_model)

In [ ]:
relevant_docs = vector_store.similarity_search("story of Captain Ahabâ€™s")

In [ ]:
def format_relevant_docs(docs):
    return [doc for doc in docs]

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant that answers questions based on the following context: {context}"), 
        ("user", "Question: {question}")
    ])

In [ ]:
# Instantiate the model
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

In [ ]:
from langchain_core.output_parsers import StrOutputParser
output_parser = StrOutputParser()

In [ ]:
# Create RAG chain
rag_chain = prompt | llm | output_parser

In [ ]:
# Invoke the RAG chain
context = format_relevant_docs(relevant_docs)
question = "What is the story of Captain Ahabâ€™s?" # Happy test case
# question = "What is the story of shubham?" # Negative test case

rag_chain.invoke(
    {"context": context, "question": question}
)